# Parte 2: Modelagem Preditiva (Machine Learning)

Nesta segunda etapa do projeto, o foco transita da análise estatística e processamento de sinais para o campo do Aprendizado de Máquina. O objetivo é desenvolver e comparar diferentes modelos de classificação capazes de diagnosticar automaticamente o estado operacional do grupo motopropulsor baseando-se na telemetria.

Para abranger diferentes abordagens de reconhecimento de padrões, o trabalho avalia quatro algoritmos distintos, cada um sob a responsabilidade de um integrante da equipe:
- **Regressão Logística**
- **Support Vector Machine (SVM)**
- **Floresta Aleatória (Random Forest)**
- **K-Nearest Neighbors (KNN)**

Esta seção avalia o modelo sob a visão da competição em que a equipe participa, convertendo os dados do algoritmo em impacto direto na pontuação da **SAE Brasil AeroDesign (Classe Micro)**.

In [ ]:
# Importações necessárias para a Parte 2 (Machine Learning)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
import seaborn as sns
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Regressão Logística
---

## 1. Extração de Características 

A extração de características ocorreu mediante uma técnica de janelamento de 50 amostras. Esta abordagem evita a correlação excessiva entre exemplos parecidos no conjunto de dados.

In [ ]:
def extrair_features_por_janela(df, tamanho_janela=50, passo=51):
    amostras = []
    # Ajustado o range para garantir que, caso o resto seja menor que 50, ele ainda processe ou ignore adequadamente
    for i in range(0, len(df) - tamanho_janela + 1, passo):
        bloco = df.iloc[i : i + tamanho_janela]
        
        features = {}
        for col in bloco.columns:
            valores = bloco[col].dropna()
            if valores.empty:
                continue
            
            # Features estatísticas básicas
            features[f"{col}_Media"] = valores.mean()
            features[f"{col}_Mediana"] = valores.median()
            features[f"{col}_DesvioPadrao"] = valores.std()
            features[f"{col}_Skew"] = valores.skew()
            features[f"{col}_Kurtosis"] = valores.kurt()
            
            # Features avançadas
            q75, q25 = np.percentile(valores, [75 ,25])
            features[f"{col}_IQR"] = q75 - q25 
            
            rms = np.sqrt(np.mean(valores**2))
            features[f"{col}_RMS"] = rms
            features[f"{col}_Energia"] = np.sum(valores**2)
            
            # Features vitais para detecção de anomalias mecânicas
            val_max = valores.max()
            val_min = valores.min()
            features[f"{col}_PicoAPico"] = val_max - val_min 
            features[f"{col}_FatorCrista"] = np.max(np.abs(valores)) / rms if rms > 0 else 0
            
        amostras.append(pd.Series(features))
        
    return amostras

## 2. Preparação e Estruturação do Dataset

In [ ]:
def processar_arquivos(lista_arquivos):
    dados = []
    for caminho, classe in lista_arquivos:
        try:
            df_i = pd.read_excel(caminho)
            colunas_proibidas = [c for c in df_i.columns if any(s in c.lower() for s in ['potencia', 'rpm', 'tempo', 'celula3'])]
            df_i = df_i.drop(columns=colunas_proibidas, errors="ignore")
            
            _, df_1, _ = transitory_stacionary(df_i)
            amostras_janeladas = extrair_features_por_janela(df_1, tamanho_janela=50, passo=51)
            
            for linha in amostras_janeladas:
                linha["Classe"] = classe
                dados.append(linha)
        except Exception:
            pass
            
    return pd.DataFrame(dados).fillna(0)